In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType
import logging

# 1. Configuration & Setup of path
CATALOG = "krishna"  
SCHEMA = "capstone_project"
TARGET_SCHEMA = f"{CATALOG}.{SCHEMA}"
BASE_PATH = "/Volumes/krishna/capstone_project/census_data"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("BronzeLayer")

class BronzeIngestor:
    def __init__(self, spark, base_path, target_schema):
        self.spark = spark
        self.base_path = base_path
        self.target_schema = target_schema
        self.schemas = {
            "population": StructType([
                StructField("State Code", IntegerType(), True),
                StructField("District Code", IntegerType(), True),
                StructField("India / State / Union Territory / District", StringType(), True),
                StructField("Name", StringType(), True),
                StructField("Year", IntegerType(), True),
                StructField("Age Group", StringType(), True),
                StructField("Gender", StringType(), True),
                StructField("Ethnic Group", StringType(), True),
                StructField("Total / Rural / Urban", StringType(), True),
                StructField("Value", LongType(), True)
            ]),
            "literacy": StructType([
                StructField("State Code", IntegerType(), True),
                StructField("District Code", IntegerType(), True),
                StructField("India / State / Union Territory / District", StringType(), True),
                StructField("Name", StringType(), True),
                StructField("Year", IntegerType(), True),
                StructField("Age Group", StringType(), True),
                StructField("Gender", StringType(), True),
                StructField("Ethnic Group", StringType(), True),
                StructField("Literacy Status", StringType(), True),
                StructField("Total / Rural / Urban", StringType(), True),
                StructField("Value", LongType(), True)
            ]),
            "employment": StructType([
                StructField("State Code", IntegerType(), True),
                StructField("District Code", IntegerType(), True),
                StructField("India / State / Union Territory / District", StringType(), True),
                StructField("Name", StringType(), True),
                StructField("Year", IntegerType(), True),
                StructField("Age Group", StringType(), True),
                StructField("Gender", StringType(), True),
                StructField("Ethnic Group", StringType(), True),
                StructField("Employment Status", StringType(), True),
                StructField("Sector", StringType(), True),
                StructField("Total / Rural / Urban", StringType(), True),
                StructField("Value", LongType(), True)
            ]),
            "region": StructType([
                StructField("State Code", IntegerType(), True),
                StructField("State Name", StringType(), True),
                StructField("State or Union Territory", StringType(), True),
                StructField("District Code", IntegerType(), True),
                StructField("District Name", StringType(), True)
            ])
        }

    def validate_regions(self, fact_df, region_df, table_name):
        """Validates that state/district codes in facts exist in the region master."""
        invalid_regions = fact_df.select("State_Code", "District_Code").distinct() \
            .filter("State_Code != 0") \
            .join(region_df, ["State_Code", "District_Code"], "left_anti")
        count = invalid_regions.count()
        if count > 0:
            logger.warning(f"Validation Alert: Found {count} orphan regional codes in {table_name}.")
        else:
            logger.info(f"Validation Success: Regional identifiers for {table_name} are consistent.")

    def ingest(self, table_name):
        try:
            file_path = f"{self.base_path}/{table_name}.csv"
            logger.info(f"Ingesting from Volume: {file_path}")
            
            df = (self.spark.read.format("csv")
                  .option("header", "true")
                  .schema(self.schemas[table_name])
                  .load(file_path))
            # Sanitizing column names for delta compatibility
            for col in df.columns:
                new_name = col.replace(" ", "_").replace("/", "_").replace("__", "_")
                df = df.withColumnRenamed(col, new_name)
            df = df.withColumn("ingestion_timestamp", F.current_timestamp())
            
            # Saving as a managed Delta table in Unity Catalog
            target_table_name = f"{self.target_schema}.bronze_{table_name}"
            df.write.format("delta").mode("overwrite").saveAsTable(target_table_name)
            logger.info(f"Successfully saved {target_table_name}")
            
            return df 
            
        except Exception as e:
            logger.error(f"Ingestion failed for {table_name}: {e}")
            raise

#3.Logic for Execution
try:
    ingestor = BronzeIngestor(spark, BASE_PATH, TARGET_SCHEMA)
    # First: ingesting the region master
    region_df = ingestor.ingest("region")
    # Second: ingesting and validating fact tables
    fact_tables = ["population", "literacy", "employment"]
    for table in fact_tables:
        fact_df = ingestor.ingest(table)
        ingestor.validate_regions(fact_df, region_df, table)

except Exception as e:
    print(f"Bronze Pipeline Failed: {e}")
    raise e

INFO:BronzeLayer:Ingesting from Volume: /Volumes/krishna/capstone_project/census_data/region.csv
INFO:BronzeLayer:Successfully saved krishna.capstone_project.bronze_region
INFO:BronzeLayer:Ingesting from Volume: /Volumes/krishna/capstone_project/census_data/population.csv
INFO:BronzeLayer:Successfully saved krishna.capstone_project.bronze_population
INFO:BronzeLayer:Ingesting from Volume: /Volumes/krishna/capstone_project/census_data/literacy.csv
INFO:BronzeLayer:Successfully saved krishna.capstone_project.bronze_literacy
INFO:BronzeLayer:Ingesting from Volume: /Volumes/krishna/capstone_project/census_data/employment.csv
INFO:BronzeLayer:Successfully saved krishna.capstone_project.bronze_employment
